# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Swaraj0204/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
!pip -q install datasets pandas pyarrow

In [9]:
from google.colab import userdata

try:
    token = userdata.get("HF_TOKEN")
    print("Token retrieved successfully!" if token else "Token is empty.")
except Exception as e:
    print("Could not access the secret:", type(e).__name__)


Token retrieved successfully!


In [3]:
from huggingface_hub import login

login(token=token)

print("Hugging Face login successful!")

Hugging Face login successful!


In [4]:
from huggingface_hub import HfApi

api = HfApi()

files = api.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

print("Available dataset files:")
for file in files:
    print(file)

Available dataset files:
.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_conte

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Data Contract — Unit of Analysis

- **One row means:** One observation of search performance for a particular combination of date and available dimensions.
- **Time window:** March 2026 (`2026-03`), used as the development month.
- **Test window:** June 2026 is reserved as a sealed test month and will not be used to develop label logic.
- **Goal:** Identify webpages that may benefit from a content refresh using historical search-performance data.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field Contract

- **Features:** Historical search impressions, clicks, click-through rate (CTR), average position, and other eligible historical metrics available in the warehouse.
- **Label:** A future search-performance outcome or a documented proxy for content-refresh priority, based on fields actually available in the dataset.
- **Context:** The observation date, available dimensions, and month used to identify the data slice.
- **Excluded:** Future performance information and any field derived from the label, because either could leak the answer into the model.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
from huggingface_hub import hf_hub_download
import pandas as pd

sample_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance_sample.parquet",
    repo_type="dataset",
    token=token
)

df = pd.read_parquet(sample_path)

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

display(df.head())

Dataset shape: (11694072, 31)

Column names:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-01,client_3ffa76342f366962,content_1a6296faee432dae,True,True,False,False,0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-06
1,2026-06-01,client_3ffa76342f366962,content_73f21e612565035a,True,True,False,False,0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-06
2,2026-06-01,client_3ffa76342f366962,content_5a5be514ff559598,True,True,False,False,0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-06
3,2026-06-01,client_3ffa76342f366962,content_05b377d0c8a5cfd8,True,True,False,False,0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-06
4,2026-06-01,client_3ffa76342f366962,content_dc34c661d63e55a9,True,True,False,False,0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-06


In [6]:
for file in files:
    if "fact_content_daily_performance/" in file:
        print(file)

fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/month=2026-04/data_0.parquet
fact_content_daily_performance/month=202

In [7]:
from huggingface_hub import hf_hub_download
import pandas as pd

# Download March 2026 data for development
march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=token
)

march_df = pd.read_parquet(march_path)

print("March dataset shape:", march_df.shape)
print("\nColumns:")
print(march_df.columns.tolist())

display(march_df.head())

March dataset shape: (9841378, 30)

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
print("March dataset shape:", march_df.shape)
print("\nColumn names:")
print(march_df.columns.tolist())

March dataset shape: (9841378, 30)

Column names:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


In [11]:
# QUERY 1: Verify the data grain

grain_check = march_df[
    ["report_date", "client_hash_id", "content_hash_id"]
]

print("Total rows:", len(grain_check))
print("Unique date-client-content combinations:",
      grain_check.drop_duplicates().shape[0])

print("Duplicate combinations:",
      grain_check.duplicated().sum())

Total rows: 9841378
Unique date-client-content combinations: 9841378
Duplicate combinations: 0


In [12]:
# QUERY 2: Verify row count and date span

march_df["report_date"] = pd.to_datetime(march_df["report_date"])

print("March row count:", len(march_df))
print("First report date:", march_df["report_date"].min())
print("Last report date:", march_df["report_date"].max())

March row count: 9841378
First report date: 2026-03-01 00:00:00
Last report date: 2026-03-31 00:00:00


In [13]:
# QUERY 3: Check data availability

for col in ["gsc_data_available", "ga4_data_available"]:
    available_rows = march_df[col].eq(True).sum()

    print(f"\n{col}")
    print("Available rows:", int(available_rows))
    print("Unavailable rows:", int(march_df[col].eq(False).sum()))
    print("Total rows:", len(march_df))


gsc_data_available
Available rows: 3611061
Unavailable rows: 6230317
Total rows: 9841378

ga4_data_available
Available rows: 413966
Unavailable rows: 6408671
Total rows: 9841378


In [14]:
# Build five candidate features for March 2026

features_df = march_df[
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_pageviews",
        "ga4_sessions",
    ]
].copy()

# Five features
features_df["day_of_month"] = features_df["report_date"].dt.day
features_df["day_of_week"] = features_df["report_date"].dt.dayofweek
features_df["log_impressions"] = __import__("numpy").log1p(
    features_df["gsc_impressions"].clip(lower=0)
)
features_df["log_clicks"] = __import__("numpy").log1p(
    features_df["gsc_clicks"].clip(lower=0)
)
features_df["log_sessions"] = __import__("numpy").log1p(
    features_df["ga4_sessions"].clip(lower=0)
)

display(features_df.head())

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,day_of_month,day_of_week,log_impressions,log_clicks,log_sessions
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,NaN,NaN,1,6,3.044522,0.000000,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,NaN,NaN,1,6,0.693147,0.000000,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,NaN,NaN,1,6,4.836282,0.693147,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,NaN,NaN,1,6,2.079442,0.000000,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,NaN,NaN,1,6,2.484907,0.000000,NaN


In [15]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Use a manageable subset for the experiment
experiment_df = march_df[
    ["gsc_clicks", "gsc_impressions", "gsc_avg_position"]
].dropna().sample(
    n=min(20000, len(march_df)),
    random_state=42
).copy()

# Label: whether the content received any clicks
y = (experiment_df["gsc_clicks"] > 0).astype(int)

# Split data into training and testing sets
indices = np.arange(len(experiment_df))
train_idx, test_idx = train_test_split(
    indices,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# BAD experiment: directly include the label-derived click count
X_leaky = experiment_df[["gsc_clicks", "gsc_impressions", "gsc_avg_position"]]

model_leaky = DecisionTreeClassifier(max_depth=3, random_state=42)
model_leaky.fit(X_leaky.iloc[train_idx], y.iloc[train_idx])

leaky_score = accuracy_score(
    y.iloc[test_idx],
    model_leaky.predict(X_leaky.iloc[test_idx])
)

# HONEST experiment: remove the label-derived column
X_honest = experiment_df[["gsc_impressions", "gsc_avg_position"]]

model_honest = DecisionTreeClassifier(max_depth=3, random_state=42)
model_honest.fit(X_honest.iloc[train_idx], y.iloc[train_idx])

honest_score = accuracy_score(
    y.iloc[test_idx],
    model_honest.predict(X_honest.iloc[test_idx])
)

print("Leaky accuracy:", round(leaky_score, 4))
print("Honest accuracy:", round(honest_score, 4))
print("Leakage inflated the score:", leaky_score > honest_score)

Leaky accuracy: 1.0
Honest accuracy: 0.9012
Leakage inflated the score: True


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# Data limitation:
# This dataset contains aggregated daily performance metrics.
# It cannot tell us the true causal impact of a content change
# on traffic or rankings. Observed performance may also depend
# on seasonality, competition, and external search-engine changes.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.